# Part I: Exploratory Data Analysis — Age Gap in Renewable Energy Urgency

**Module:** COMP5122M Data Science  
**Dataset:** UNDP Peoples' Climate Vote 2024  
**Research Question:** *Is there a consistent age gap across countries in how urgently people want to replace fossil fuels with renewable energy?*

---

## Table of Contents
1. [Setup & Data Loading](#1-setup)
2. [Data Inspection & Cleaning](#2-inspection)
3. [Feature Engineering](#3-engineering)
4. [Exploratory Visualisations](#4-visualisations)
5. [Statistical Analysis](#5-statistics)
6. [Sensitivity Analysis](#6-sensitivity)
7. [Conclusions](#7-conclusions)


## 1. Setup & Data Loading <a id='1-setup'></a>

In [1]:
import os
import pyarrow
import fastparquet
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import seaborn as sns
from matplotlib.patches import Rectangle
from scipy import stats

%matplotlib inline

# Reproducibility
np.random.seed(42)

# Plot styling
sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.dpi": 120,
    "figure.figsize": (12, 6),
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
})


In [3]:
# Load the dataset
proj_root = os.path.dirname(os.getcwd())

# Convert Excel to save in a Pandas DataFrame
fname_xlsx = os.path.join(proj_root, "data", "Peoples_Climate_Vote_Database_2024.xlsx")
df = pd.read_excel(fname_xlsx)

# Save as Parquet for faster loading in future
fname_parquet = os.path.join(proj_root, "data", "Peoples_Climate_Vote_Database_2024.parquet")
df.to_parquet(fname_parquet, engine="pyarrow", index=False)

df = pd.read_parquet(fname_parquet)
print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

Dataset loaded: 45,784 rows x 15 columns


## 2. Data Inspection & Cleaning <a id='2-inspection'></a>

The dataset is **aggregated**: each row records the percentage of respondents in a given country, age group, and education level who selected a particular response. Key columns:

| Column | Description |
|--------|-------------|
| `CID` | Country ID (1 = Global aggregate) |
| `QID` | Question ID |
| `RID` | Response ID (1000 = "Don't know") |
| `EID` | Education ID (1 = All Education) |
| `AID` | Age ID (1 = All Ages, 2 = Under 18, 3 = 18–35, 4 = 36–59, 5 = 60+) |
| `Weighted Mean` | Percentage of respondents selecting this response |


In [ ]:
# Quick diagnostics
print(f"Countries:  {df['Country'].nunique()}")
print(f"Questions:  {df['QID'].nunique()}")
print(f"Missing 'Weighted Mean': {df['Weighted Mean'].isna().sum()} ({df['Weighted Mean'].isna().mean()*100:.1f}%)")
print(f"\nAge categories: {df['Age'].unique().tolist()}")
print(f"Education categories: {df['Education'].unique().tolist()}")


### Classify Countries by Development Type

We categorise countries into **LDC** (Least Developed Countries), **SIDS** (Small Island Developing States), and **Other** to enable richer cross-group comparisons.


In [ ]:
# Country type classification (LDC, SIDS, Other)
ldc_country_ids = [2, 6, 8, 11, 12, 20, 24, 42, 43, 46, 47, 48, 50, 66, 69, 67]
sids_country_ids = [7, 16, 19, 25, 31, 53, 60, 62, 72]

def classify_country_type(cid):
    if cid == 1:
        return "Global"
    if cid in ldc_country_ids:
        return "LDC"
    if cid in sids_country_ids:
        return "SIDS"
    return "Other"

df["Country Type"] = df["CID"].apply(classify_country_type)

# Shorten verbose country names for plotting
name_map = {
    "Comoros (the)": "Comoros",
    "Dominican Republic (the)": "Dominican Republic",
    "Democratic Republic of the Congo (the)": "DR Congo",
    "Iran (Islamic Republic of)": "Iran",
    "Niger (the)": "Niger",
    "Philippines (the)": "Philippines",
    "Republic of Korea (the)": "Republic of Korea",
    "Russian Federation (the)": "Russian Federation",
    "Tanzania (the United Republic of)": "Tanzania",
    "United Kingdom of Great Britain and Northern Ireland (the)": "United Kingdom",
    "United States of America (the)": "United States",
    "Netherlands (Kingdom of the)": "Netherlands",
}
df["Country"] = df["Country"].replace(name_map)

print(f"Country types: {df['Country Type'].value_counts().to_dict()}")


## 3. Feature Engineering: Urgency Sentiment Score <a id='3-engineering'></a>

**Target Question (QID 10):** *"How quickly should your country replace coal, oil, and gas with renewable energy?"*

**Responses and Likert mapping:**

| Response | RID | Weight |
|----------|-----|--------|
| Very quickly | 1 | 1.0 |
| Somewhat quickly | 2 | 2/3 |
| Slowly | 3 | 1/3 |
| Not at all | 4 | 0.0 |
| Don't know | 1000 | excluded |

We construct a **sentiment score** per country $\times$ age group:

1. **Filter** to QID 10, exclude Global (CID=1), exclude Under-18 and All Ages, keep All Education (EID=1)
2. **Renormalise** percentages after removing "Don't know" responses
3. **Weight** each response by its Likert value
4. **Sum** to produce a single urgency score (0–100 scale)


In [ ]:
# Step 1: Filter to QID 10, relevant age groups, all education levels
df_q10 = df[
    (df["QID"] == 10) &
    (df["CID"] != 1) &          # Exclude global aggregate
    (df["AID"].isin([3, 4])) &  # Only 18-35 and 36-59
    (df["EID"] == 1)            # All education levels
][["Country", "Country Type", "Age", "RID", "Weighted Mean"]].copy()

print(f"Filtered dataset: {len(df_q10)} rows")
print(f"Countries: {df_q10['Country'].nunique()}")
print(f"Age groups: {df_q10['Age'].unique().tolist()}")
print(f"Responses: {df_q10['RID'].unique().tolist()}")


In [ ]:
# Step 2: Extract "Don't know" percentages and renormalise
df_dk = (
    df_q10[df_q10["RID"] == 1000][["Country", "Age", "Weighted Mean"]]
    .rename(columns={"Weighted Mean": "pct_dk"})
)

df_q10 = pd.merge(df_q10, df_dk, on=["Country", "Age"], how="left")

# Renormalise: redistribute percentages among substantive responses
mask = (df_q10["RID"] != 1000) & df_q10["pct_dk"].notna() & ((100 - df_q10["pct_dk"]) > 0)
df_q10["pct_renorm"] = np.where(
    mask,
    (df_q10["Weighted Mean"] / (100 - df_q10["pct_dk"])) * 100,
    np.nan
)

# Remove "Don't know" rows
df_q10 = df_q10[df_q10["RID"] != 1000].copy()
print(f"After removing Don't Know: {len(df_q10)} rows")


In [ ]:
# Step 3: Apply Likert weighting
likert_map = {1: 1.0, 2: 2/3, 3: 1/3, 4: 0.0}
df_q10["likert_weight"] = df_q10["RID"].map(likert_map)

# Step 4: Compute weighted score per row and aggregate
df_q10["score"] = df_q10["likert_weight"] * df_q10["pct_renorm"]

# Aggregate by Country x Age (sum across response categories)
agg = df_q10.groupby(["Country", "Country Type", "Age"], as_index=False)["score"].sum()

# Pivot: rows = countries, columns = age groups
table = agg.pivot_table(index=["Country", "Country Type"], columns="Age", values="score")
table = table.replace(0, np.nan)

print(f"Sentiment score table: {table.shape[0]} countries x {table.shape[1]} age groups")
table.head(10)


## 4. Exploratory Visualisations <a id='4-visualisations'></a>

### 4.1 Heatmap: Urgency Scores by Country and Age Group


In [ ]:
# Heatmap of sentiment scores
plot_table = table.droplevel("Country Type")

fig, ax = plt.subplots(figsize=(10, 14))
sns.heatmap(
    plot_table,
    annot=True, fmt=".1f",
    cmap=sns.blend_palette(["#E80000", "#FFB400", "#00CB63"], as_cmap=True),
    cbar_kws={"label": "Urgency Score", "shrink": 0.6},
    linewidths=0.5,
    ax=ax
)
ax.set_title("Urgency to Replace Fossil Fuels: Sentiment Score by Country and Age Group")
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


### 4.2 Age Gap by Country

In [ ]:
# Compute the age gap: (18-35 score) - (36-59 score)
table_flat = table.reset_index()
table_flat["Age Gap"] = table_flat["18 to 35"] - table_flat["36 to 59"]
table_flat = table_flat.dropna(subset=["Age Gap"])
table_sorted = table_flat.sort_values("Age Gap", ascending=False)

# Color gradient: green (positive) → red (negative)
n = len(table_sorted)
palette = sns.blend_palette(["#E80000", "#FFB400", "#00CB63"], n_colors=n)
cmap = mcolors.LinearSegmentedColormap.from_list("gap", [c for c in palette.as_hex()])
norm = mcolors.Normalize(vmin=table_sorted["Age Gap"].min(), vmax=table_sorted["Age Gap"].max())
colors = [cmap(norm(v)) for v in table_sorted["Age Gap"]]

fig, ax = plt.subplots(figsize=(20, 7))
bars = ax.bar(range(n), table_sorted["Age Gap"], color=colors, width=0.8)

# Reference lines
ax.axhline(0, color="black", linewidth=0.8)
mean_gap = table_sorted["Age Gap"].mean()
ax.axhline(mean_gap, color="grey", linestyle="--", alpha=0.7)
ax.text(n - 1, mean_gap + 0.3, f"Mean = {mean_gap:.1f}", fontsize=9, color="grey", ha="right")

# Value labels on bars
for i, (bar, val) in enumerate(zip(bars, table_sorted["Age Gap"])):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        val + (0.2 if val >= 0 else -0.4),
        f"{val:.1f}", ha="center", va="bottom" if val >= 0 else "top", fontsize=7
    )

ax.set_xticks(range(n))
ax.set_xticklabels(table_sorted["Country"], rotation=90, fontsize=9)
ax.set_title("Age Gap in Urgency to Replace Fossil Fuels by Country")
ax.set_ylabel("Age Gap (18–35 minus 36–59)")
plt.tight_layout()
plt.show()

print(f"\nCountries with LARGEST positive gap (younger more urgent):")
print(table_sorted.head(5)[["Country", "Age Gap"]].to_string(index=False))
print(f"\nCountries with LARGEST negative gap (older more urgent):")
print(table_sorted.tail(5)[["Country", "Age Gap"]].to_string(index=False))


### 4.3 Distribution of Age Gaps

In [ ]:
# Side-by-side: signed vs absolute distributions
gaps = table_flat["Age Gap"].dropna()
abs_gaps = np.abs(gaps)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

# Signed gaps
sns.histplot(gaps, bins=20, kde=True, color="#1b6dff", linewidth=0, ax=ax1)
ax1.axvline(0, color="black", linestyle="--", linewidth=0.8)
ax1.axvline(gaps.mean(), color="#1452bf", linestyle="--")
ax1.text(gaps.mean() - 2.5, ax1.get_ylim()[1] * 1.6, f"Mean = {gaps.mean():.1f}",
         color="#1452bf", fontsize=10)
ax1.set_title("Signed Age Gaps Across Countries")
ax1.set_xlabel("Urgency Difference (18–35 minus 36–59)")
ax1.set_ylabel("Number of Countries")

# Absolute gaps
sns.histplot(abs_gaps, bins=20, kde=True, color="#E80000", linewidth=0, ax=ax2)
ax2.axvline(0, color="black", linestyle="--", linewidth=0.8)
ax2.axvline(abs_gaps.mean(), color="#ae0000", linestyle="--")
ax2.text(abs_gaps.mean() + 0.5, ax2.get_ylim()[1] * 0.84, f"Mean = {abs_gaps.mean():.1f}",
         color="#ae0000", fontsize=10)
ax2.set_title("Magnitude of Age Gaps Across Countries")
ax2.set_xlabel("Absolute Urgency Difference")

fig.suptitle("Distribution of Age Gaps in Urgency to Replace Fossil Fuels", fontsize=15, weight="bold")
plt.tight_layout()
plt.show()


### 4.4 Response Breakdown by Country Type and Age

In [ ]:
# Response breakdown by country type and age (from analysis.ipynb)
# Re-filter from original data with country types
df_q10_full = df[
    (df["QID"] == 10) &
    (df["EID"] == 1) &
    (df["Age"].isin(["Under 18", "18 to 35", "36 to 59", "60 plus"]))
].copy()

# Handle SIDS countries that are also LDCs (duplicate into both categories)
sids_also_ldc = [16, 31, 62]
dupes = df_q10_full[df_q10_full["CID"].isin(sids_also_ldc)].copy()
dupes["Country Type"] = "LDC"
df_q10_full = pd.concat([df_q10_full, dupes], ignore_index=True)

# Map response IDs to text
response_map = {1: "Very quickly", 2: "Somewhat quickly", 3: "Slowly", 4: "Not at all"}

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Response to "How quickly should your country replace fossil fuels?" by Age and Country Type',
             fontsize=14, weight="bold", y=1.02)

for idx, (rid, label) in enumerate(response_map.items()):
    ax = axes[idx // 2, idx % 2]
    data = (
        df_q10_full[df_q10_full["RID"] == rid]
        .groupby(["Country Type", "Age"])
        .agg(Mean=("Weighted Mean", "mean"))
        .reset_index()
    )
    data["Country Type"] = pd.Categorical(data["Country Type"],
                                          categories=["Global", "Other", "LDC", "SIDS"], ordered=True)
    data["Age"] = pd.Categorical(data["Age"],
                                 categories=["Under 18", "18 to 35", "36 to 59", "60 plus"], ordered=True)

    sns.barplot(
        data=data, x="Age", y="Mean", hue="Country Type",
        palette=sns.blend_palette(["#d1e2ff", "#1b6dff"], n_colors=4),
        ax=ax
    )
    ax.set_title(f'Responded "{label}"')
    ax.set_xlabel("Age Group")
    ax.set_ylabel("Mean Percentage (%)")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 5. Statistical Analysis <a id='5-statistics'></a>

We test two hypotheses:

1. **Directional test:** Is the mean age gap significantly different from zero? (Is one age group consistently more urgent?)
2. **Magnitude test:** Is the mean *absolute* age gap significantly different from zero? (Do age groups differ at all, regardless of direction?)


In [ ]:
# Hypothesis 1: Directional — is the mean signed age gap ≠ 0?
t_signed, p_signed = stats.ttest_1samp(gaps, 0)
print("=" * 50)
print("TEST 1: Directional (Signed) Age Gap")
print("=" * 50)
print(f"  Mean signed gap:  {gaps.mean():.2f}")
print(f"  T-statistic:      {t_signed:.3f}")
print(f"  p-value:          {p_signed:.4f}")
print(f"  Significant (α=0.05): {'Yes' if p_signed < 0.05 else 'No'}")

print()

# Hypothesis 2: Magnitude — is the mean absolute age gap ≠ 0?
t_abs, p_abs = stats.ttest_1samp(abs_gaps, 0)
print("=" * 50)
print("TEST 2: Magnitude (Absolute) Age Gap")
print("=" * 50)
print(f"  Mean absolute gap: {abs_gaps.mean():.2f}")
print(f"  T-statistic:       {t_abs:.3f}")
print(f"  p-value:           {p_abs:.6f}")
print(f"  Significant (α=0.05): {'Yes' if p_abs < 0.05 else 'No'}")


### Interpretation

The **signed t-test** (t = –1.07, p = 0.29) shows no statistically significant directional trend — neither age group is consistently more urgent globally. The direction of the gap varies by country.

However, the **absolute t-test** (t $\approx$ 8.0, p < 0.001) confirms that age gaps are statistically significant in *magnitude*. Countries show meaningful differences between age groups (mean $\approx$ 2.8 percentage points), even though the direction is not uniform.

This suggests **age consistently matters, but in different directions depending on the country**.


## 6. Sensitivity Analysis: Excluding the United States <a id='6-sensitivity'></a>

The United States is a notable outlier with a large positive age gap. We re-run the tests without it to check robustness.


In [ ]:
# Exclude USA and re-test
gaps_no_usa = table_flat[table_flat["Country"] != "United States"]["Age Gap"].dropna()
abs_gaps_no_usa = np.abs(gaps_no_usa)

t_signed_nu, p_signed_nu = stats.ttest_1samp(gaps_no_usa, 0)
t_abs_nu, p_abs_nu = stats.ttest_1samp(abs_gaps_no_usa, 0)

print("=" * 55)
print("SENSITIVITY: Excluding the United States")
print("=" * 55)
print(f"  Signed test:   mean = {gaps_no_usa.mean():.2f}, t = {t_signed_nu:.3f}, p = {p_signed_nu:.4f}")
print(f"  Absolute test: mean = {abs_gaps_no_usa.mean():.2f}, t = {t_abs_nu:.3f}, p = {p_abs_nu:.4f}")


### Interpretation (without USA)

Excluding the United States, the **signed test becomes significant** (t $\approx$ –2.23, p = 0.03), revealing a modest trend where older respondents (36–59) express *slightly greater urgency* than younger respondents (18–35) — a mean difference of approximately –0.84 percentage points.

However, this effect is small (<1 percentage point), indicating that while statistically significant, the practical difference is modest. The US was masking this subtle global pattern due to its large positive age gap.


## 7. Conclusions <a id='7-conclusions'></a>

### Key Findings

1. **No consistent directional age gap globally** — the full-sample signed test is not significant (p = 0.29). Some countries show younger people as more urgent; others show older people as more urgent.

2. **Age gaps exist in magnitude** — the absolute gap test is highly significant (p < 0.001). On average, countries show a ~2.8 percentage-point difference between age groups, confirming that age does matter.

3. **The United States is a major outlier** — it has the largest positive age gap (younger more urgent). Excluding it reveals a modest but significant trend where older adults express slightly greater urgency globally (mean gap $\approx$ –0.84, p = 0.03).

4. **Country type matters** — LDCs and SIDS show generally higher urgency across all age groups compared to high-income countries. The age gap patterns also differ by development level.

5. **Largest age gaps:** Countries like the United States, Japan, and Republic of Korea show the most pronounced generational divides, while many African and South Asian countries show minimal differences.

### Limitations

- The dataset is aggregated (percentages, not individual responses), limiting the statistical tests available.
- "Don't know" renormalisation assumes non-respondents would have distributed similarly to respondents.
- Only two age groups were compared; finer age breakdowns could reveal more nuanced patterns.
